In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

# Load Data 

In [ ]:
from data import TextDataloaderConfig, create_text_dataloaders

CFG = TextDataloaderConfig(
    preset_name="wikitext2",      # wikitext2, tinystories, ag_news, imdb, minipile
    block_size=128,
    batch_size=8,
    max_tokenizer_documents=5000,
    max_train_documents=2000,
    max_validation_documents=500,
    tokenizer_path="data/cache/tokenizers/wikitext2.json",
)

train_loader, val_loader, tokenizer = create_text_dataloaders(
    cfg=CFG,
    use_mtp=False,
)

In [ ]:
from data.syntethic_long_context_retrieval import *

CFG = SyntheticRetrievalConfig(
    # Dataset size
    num_train_examples=50_000,
    num_val_examples=5_000,

    # Sequence length
    block_size=64,              # debe ser <= model.config.max_seq_len
    min_filler_tokens=8,
    max_filler_tokens=32,

    # Task structure
    num_keys_per_example=4,
    vocab_filler_size=68,      
    num_key_types=64,
    num_value_types=64,

    # DataLoader
    batch_size=32,
    num_workers=0,

    # Reproducibility
    seed=42)

train_loader, val_loader, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=CFG,
        use_mtp=False,)

x, y = next(iter(train_loader))

Synthetic tokenizer vocab size: 954


---

In [ ]:
from src.mini_deepseek_class import * 

cfg = DeepSeekV4LMConfig(
    vocab_size=216,
    d_model=64,
    n_layers=4,
    max_seq_len=64,
    pad_token_id=0,
    ignore_index=-100,

    # ========================================================
    # Attention: hybrid CSA/HCA baseline
    # Pattern:
    #   layer 0 -> CSA
    #   layer 1 -> HCA
    #   layer 2 -> CSA
    #   layer 3 -> HCA
    # ========================================================
    attention_type="hybrid",
    attention_pattern=("csa", "hca"),

    n_heads=4,
    head_dim=16,
    rotary_dim=16,

    attention_dropout=0.0,
    residual_dropout=0.0,
    use_attention_bias=False,
    use_rope=True,
    rope_theta=10000.0,

    # ========================================================
    # CSA config
    # ========================================================
    compression_factor=4,
    top_k_blocks=2,
    window_size=8,
    indexer_dim=8,
    n_indexer_heads=2,
    query_compression_dim=16,

    use_attention_sink=True,
    use_grouped_output_projection=True,
    output_projection_groups=2,
    use_indexer_score_bias=False,
    use_separate_local_kv=True,

    # ========================================================
    # HCA config
    # HCA should compress more aggressively than CSA.
    # With max_seq_len=64, hca_compression_factor=8 is a sane
    # mini baseline. 16 also works but gives very few global blocks.
    # ========================================================
    hca_compression_factor=8,

    # ========================================================
    # FFN: DeepSeekMoE
    # ========================================================
    ffn_type="moe",
    num_experts=4,
    top_k_experts=2,

    expert_hidden_dim=128,
    expert_expansion_factor=4.0,
    expert_multiple_of=1,

    shared_experts=1,
    shared_hidden_dim=128,
    shared_expansion_factor=4.0,

    router_type="learned",
    router_score_fn="sqrt_softplus",
    normalize_topk_weights=True,
    topk_weight_scale=1.0,
    router_jitter_noise=0.0,
    hash_routing_stride=1,

    routed_scale=1.0,
    shared_scale=1.0,

    balance_loss_weight=0.01,
    sequence_balance_loss_weight=0.01,

    # ========================================================
    # mHC
    # ========================================================
    use_mhc=True,
    n_hc=4,
    mhc_sinkhorn_iters=20,
    mhc_eps=1e-6,
    mhc_dynamic=True,
    mhc_expand_mode="first",
    mhc_collapse_mode="readout",

    mhc_use_log_sinkhorn=False,
    mhc_sinkhorn_fp32=True,
    mhc_init_alpha=1e-3,
    mhc_alpha_max=1.0,
    mhc_bounded_alpha=True,

    # ========================================================
    # MTP
    # ========================================================
    use_mtp=True,
    mtp_depth=2,
    mtp_hidden_dim=64,
    use_mtp_transform=True,
    mtp_activation="silu",
    mtp_dropout=0.0,
    mtp_loss_weight=0.3,
    mtp_tie_with_lm_head=False,
    mtp_depth_loss_weights=None,
    mtp_validate_label_range=True,

    # ========================================================
    # Embedding / norm / init
    # ========================================================
    embedding_dropout=0.0,
    scale_embeddings=False,
    tie_word_embeddings=True,

    rms_norm_eps=1e-6,
    init_std=0.02,

    # ========================================================
    # Loss semantics
    #   input_ids = ids[:-1]
    #   labels    = ids[1:]
    # ========================================================
    labels_are_shifted=True,
    ignore_pad_token_in_loss=True,)

model = DeepSeekV4LM(cfg)
model.to('cuda')

for i, block in enumerate(model.blocks):
    print(i, block.attention_type, type(block.attention).__name__)

0 csa CSAAttention
1 hca HCAAttention
2 csa CSAAttention
3 hca HCAAttention


In [ ]:
from training.train_deepseek import * 

result = train_deepseekv4(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,

    optimizer_type="muon_adamw",
    learning_rate=3e-4,
    min_learning_rate=3e-5,
    weight_decay=0.1,

    epochs=15,
    max_batches_per_epoch=200,
    log_every=20,
    module_metrics_every=0, # Para imrpimir al final solo
    verbose=1,

    eval_every=1,
    eval_max_batches=50,
    eval_use_ema=True,
    eval_preview=True,
    eval_preview_max_context_tokens=48,
    eval_preview_max_new_tokens=24,
    eval_preview_temperature=0.0,

    use_ema=True,
    ema_decay=0.999,
    ema_device="cpu",

    ckpt_dir="checkpoints/deepseekv4_mini",
    run_name="deepseekv4_mini_muon",
    monitor_name="eval_loss",
    monitor_mode="min",

    drive_ckpt_dir=None,
)

Hybrid Muon + AdamW parameter groups
Muon tensors          : 108
AdamW decay tensors   : 10
AdamW no-decay tensors: 127
--------------------------------------------------------------------------------
Muon params           : 548,608
AdamW params          : 94,556
Total trainable params: 643,164

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
DeepSeek-V4 run: deepseekv4_mini_muon
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Device    : cuda | AMP: True (bf16 -> torch.bfloat16)
Optimizer : muon_adamw | EMA: 0.999000 | grad_clip: 1.0 | grad_accum_steps: 1
Schedule  : epochs=15 | start_epoch=0 | global_step=0 | total_steps=3000 | warmup_steps=500
Monitor   : eval_loss (min) | best_metric=None
Model     : layers=4 | d_model=64 | attention=hybrid | ffn=moe | mHC=True | MTP=True
Eval      : enabled | eval_every=1 | eval_max_batches=50
──────────────────────────────

KeyboardInterrupt: 